In [1]:
# Cell 1: Import libraries and aggregate player statistics from StatsBomb open data

import pandas as pd
import numpy as np
from statsbombpy import sb
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

print("Fetching World Cup 2022 event data for player profiling...")

# 1. Fetch all event data for World Cup 2022
events = sb.competition_events(
    country="International",
    division="FIFA World Cup",
    season="2022",
    gender="male"
)

# 2. Filter events with valid player information
player_events = events.dropna(subset=['player']).copy()

# 3. Aggregate raw performance metrics per player
player_stats = player_events.groupby('player').agg(
    team=('team', 'first'),
    total_shots=('type', lambda x: (x == 'Shot').sum()),
    total_goals=('shot_outcome', lambda x: (x == 'Goal').sum()),
    total_passes=('type', lambda x: (x == 'Pass').sum()),
    completed_passes=('pass_outcome', lambda x: x.isnull().sum()), # In StatsBomb, null outcome means successful pass
    dribbles=('type', lambda x: (x == 'Dribble').sum()),
    interceptions=('type', lambda x: (x == 'Interception').sum())
).reset_index()

# 4. Filter active players to avoid sample noise (minimum 20 passes or 3 shots)
active_players = player_stats[(player_stats['total_passes'] >= 20) | (player_stats['total_shots'] >= 3)].copy()

print(f"Dataset ready! Total active players profiled: {len(active_players)}")

# Display the first 10 active player profiles
active_players.head(10)

Fetching World Cup 2022 event data for player profiling...


C:\Users\jpazu\anaconda3\envs\ProjetoFutebolPredictions\Lib\site-packages\statsbombpy\api_client.py:27: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Dataset ready! Total active players profiled: 562


,player,team,total_shots,total_goals,total_passes,completed_passes,dribbles,interceptions
0,Aaron Mooy,Australia,1,0,219,628,5,3
1,Aaron Ramsey,Wales,1,0,110,433,9,1
2,Abdelhamid Sabiri,Morocco,4,1,58,230,1,5
3,Abdelkarim Hassan Al Haj Fadlalla,Qatar,6,0,158,491,3,3
6,Abdou Diallo,Senegal,1,0,183,492,0,1
8,Abdul Rahman Baba,Ghana,0,0,103,302,5,1
9,Abdulaziz Hatem Mohammed Abdullah,Qatar,1,0,91,287,1,2
10,Abdulelah Al Amri,Saudi Arabia,1,0,101,285,0,4
11,Abdulelah Saad Hameed Al-Malki,Saudi Arabia,2,0,86,286,0,3
14,Abolfazl Jalali,Iran,0,0,29,81,2,0


In [4]:
# Cell 2 (Updated): Scale features, compute Cosine Similarity, and search with Partial Name Matching

# 1. Select numeric metrics for similarity profiling
feature_cols = ['total_shots', 'total_goals', 'total_passes', 'completed_passes', 'dribbles', 'interceptions']

# 2. Set 'player' as the DataFrame index
features_df = active_players.set_index('player')[feature_cols]

# 3. Standardize features (Mean = 0, Variance = 1)
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_df)

# 4. Compute pairwise Cosine Similarity matrix
similarity_matrix = cosine_similarity(scaled_features)
similarity_df = pd.DataFrame(similarity_matrix, index=features_df.index, columns=features_df.index)

# 5. Helper function supporting PARTIAL NAME MATCHING (e.g. "Messi", "Mbappe", "Fernandez")
def get_similar_players(query_name, top_n=5):
    # Search for player names containing the query string (case-insensitive)
    matches = [p for p in similarity_df.index if query_name.lower() in p.lower()]
    
    if not matches:
        return f"No player matching '{query_name}' was found in the active dataset."
    
    # Pick the first matched full name
    target_player = matches[0]
    print(f"--- Scouting Profile Match for: '{target_player}' ---")
    
    # Get top N similar players excluding the query player itself
    similar_scores = similarity_df[target_player].sort_values(ascending=False)[1:top_n+1]
    
    # Return formatted summary table
    results = pd.DataFrame({
        'Similar Player': similar_scores.index,
        'Similarity Match (%)': (similar_scores.values * 100).round(2)
    }).reset_index(drop=True)
    
    return results

print("Smart Cosine Similarity Engine ready!")

# Test 
get_similar_players("Ronaldo", top_n=5)

Smart Cosine Similarity Engine ready!
--- Scouting Profile Match for: 'Cristiano Ronaldo dos Santos Aveiro' ---


,Similar Player,Similarity Match (%)
0,Marco Asensio Willemsen,98.78
1,Lautaro Javier Martínez,94.14
2,Marko Livaja,93.24
3,Harry Kane,91.78
4,Mitchell Thomas Duke,91.66


In [5]:
# Cell 3: Export active player statistics to a CSV file for Streamlit deployment
active_players.to_csv('player_stats.csv', index=False)
print("Player statistics exported successfully as 'player_stats.csv'!")

Player statistics exported successfully as 'player_stats.csv'!
